# ProtoFormer — PASCAL-5i fold 0
1. cfg
2. drive
3. repo
4. deps
5. voc
6. aug
7. sam2 ckpt
8. imports
9. train 1shot
10. train 5shot
11. train_epoch fn
12. test
13. save


In [ ]:
# 1
import os, sys, json, subprocess, shutil, tarfile, zipfile
from pathlib import Path

DRIVE_FOLDER = "ProtoFormer_out"
REPO = "/content/ProtoFormer"
GIT_URL = "https://github.com/leilei2018/ProtoFormer.git"
BRANCH = "main"
DATA = Path("/content/data")
SAM2_CKPT = "/content/sam2_hiera_l.pth"
SAM2_DRIVE_SRC = "/content/drive/MyDrive/project/checkpoints/sam2_hiera_large.pt"
AUG_URL = "https://www.dropbox.com/s/oeu149j8qtbs1x0/SegmentationClassAug.zip?raw=1"

CFG = dict(
    benchmark="pascal", fold=0,
    shots=[1, 5],
    bsz=32, lr=1e-3, niter=60, nworker=4,
    reduce_dim=64,
    amp="auto", logpath="pascal_fold0", patience=20, val_interval=3,
)
print(CFG)


In [ ]:
# 2
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
OUT = Path(f"/content/drive/MyDrive/{DRIVE_FOLDER}")
OUT.mkdir(parents=True, exist_ok=True)
print("OUT:", OUT)


In [ ]:
# 3
if not Path(REPO).exists():
    subprocess.run(["git", "clone", "-b", BRANCH, GIT_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", f"origin/{BRANCH}"], check=True)
    subprocess.run(["git", "-C", REPO, "clean", "-fd"], check=True)
os.chdir(REPO)
assert Path(REPO, "train.py").exists()
print("repo OK")


In [ ]:
# 4
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
import torch, torchvision
print(torch.__version__, torchvision.__version__)


In [ ]:
# 5
from datasets import load_dataset
from PIL import Image

voc = DATA / "VOC2012"
jpg_dir = voc / "JPEGImages"
seg_dir = voc / "SegmentationClassAug"
img_dir = voc / "SegmentationClass"

# Download dari HuggingFace (parquet, bukan tar Oxford yang gagal di Colab).
# segmentation_aug = 10582 train (mask SBD = SegmentationClassAug) / 1449 val (mask resmi).
# segmentation = 1464/1449 mask resmi (SegmentationClass). Keduanya menutup
# kebutuhan PASCAL-5i: split protoformer (11394 train / 346 val fold0) hanya
# butuh file jpg+png per id, bukan tar VOC utuh.
if not jpg_dir.exists() or not seg_dir.exists():
    ds = load_dataset("shijli/voc2012", "segmentation_aug")
    seen = set()
    for split_name in ("train", "validation"):
        sp = ds[split_name]
        for i in range(len(sp)):
            row = sp[i]
            img_id = row["id"]
            if img_id in seen:
                continue
            seen.add(img_id)
            row["image"].convert("RGB").save(str(jpg_dir / f"{img_id}.jpg"), "JPEG", quality=95)
            row["mask"].save(str(seg_dir / f"{img_id}.png"), "PNG")
    # mask resmi val (SegmentationClass) untuk val fold 1449
    ds2 = load_dataset("shijli/voc2012", "segmentation")
    for split_name in ("train", "validation"):
        sp = ds2[split_name]
        for i in range(len(sp)):
            row = sp[i]
            row["mask"].save(str(img_dir / f"{row['id']}.png"), "PNG")
    del ds, ds2

exp_jpg = 10582 + 1449
exp_aug = 10582 + 1449
n_jpg = len(list(jpg_dir.glob("*.jpg"))) if jpg_dir.exists() else 0
n_aug = len(list(seg_dir.glob("*.png"))) if seg_dir.exists() else 0
n_off = len(list(img_dir.glob("*.png"))) if img_dir.exists() else 0
assert n_jpg >= exp_jpg - 200, f"JPEGImages kurang: {n_jpg}/{exp_jpg}"
assert n_aug >= exp_aug - 200, f"SegmentationClassAug kurang: {n_aug}/{exp_aug}"
assert n_off >= 2913 - 100, f"SegmentationClass kurang: {n_off}/2913"
print("VOC OK", n_jpg, "jpg", n_aug, "aug-png", n_off, "official-png")


In [ ]:
# 6
aug = DATA / "VOC2012/SegmentationClassAug"
if not list(aug.glob("*.png")):
    subprocess.run(["wget", "-O", "/content/aug.zip", AUG_URL], check=True)
    assert Path("/content/aug.zip").stat().st_size > 1 * 1024 * 1024
    with zipfile.ZipFile("/content/aug.zip") as z:
        z.extractall("/content/aug", filter="data")
    src = next(Path("/content/aug").rglob("SegmentationClassAug"))
    if aug.exists():
        shutil.rmtree(aug)
    shutil.move(str(src), str(aug))
n = len(list(aug.glob("*.png")))
assert n > 1000, f"SegmentationClassAug rusak: {n} png"
print("SegmentationClassAug OK", n, "png")


In [ ]:
# 7
if Path(SAM2_DRIVE_SRC).exists():
    shutil.copy(SAM2_DRIVE_SRC, SAM2_CKPT)
assert Path(SAM2_CKPT).exists() and Path(SAM2_CKPT).stat().st_size > 100 * 1024 * 1024, \
    f"SAM2 checkpoint rusak: {Path(SAM2_CKPT).stat().st_size if Path(SAM2_CKPT).exists() else 'missing'} byte"
print("SAM2 OK", Path(SAM2_CKPT).stat().st_size // 1024 // 1024, "MB")


In [ ]:
# 8
sys.path.insert(0, f"{REPO}/model")
import torch
from common.amp import get_device, resolve_amp_dtype, make_scaler, amp_context
from common.logger import Logger, AverageMeter
from common.evaluation import Evaluator
from common import utils
from data.dataset import FSSDataset
from model.fewshot import FewShotSeg as ProtoFormer
import torch.optim as optim

device = get_device()
amp = resolve_amp_dtype(device, CFG["amp"])
scaler = make_scaler(amp)
print(device, amp)
Evaluator.initialize()
FSSDataset.initialize(img_size=473, datapath=str(DATA))


In [ ]:
# 9
shot = 1
print(f"=== pascal fold0 shot={shot} ===")
model = ProtoFormer(shot=shot, reduce_dim=CFG["reduce_dim"], checkpoint_path=SAM2_CKPT).to(device)
opt = optim.Adam(model.parameters(), lr=CFG["lr"])
tr = FSSDataset.build_dataloader(CFG["benchmark"], CFG["bsz"], CFG["nworker"], CFG["fold"], "trn", shot=shot)
va = FSSDataset.build_dataloader(CFG["benchmark"], 1, CFG["nworker"], CFG["fold"], "val", shot=shot)
hist = {"epoch": [], "trn_loss": [], "trn_miou": [], "val_miou": []}
best = -1.0
no_imp = 0
for e in range(CFG["niter"]):
    tl, tm, tf = train_epoch(e, model, tr, opt, True, amp, scaler)
    hist["epoch"].append(e); hist["trn_loss"].append(float(tl)); hist["trn_miou"].append(float(tm))
    hist["val_miou"].append(None)
    if (e + 1) % CFG.get("val_interval", 3) != 0 and e != CFG["niter"] - 1:
        print(f"epoch {e}: trn_loss={float(tl):.4f} trn_mIoU={float(tm):.2f}")
        continue
    with torch.no_grad():
        vl, vm, vf = train_epoch(e, model, va, opt, False, amp, None)
    hist["val_miou"][-1] = float(vm)
    if float(vm) > best:
        best = float(vm)
        no_imp = 0
        torch.save(model.state_dict(), OUT / f"best_model_{shot}shot.pt")
    else:
        no_imp += 1
        if CFG.get("patience", 20) > 0 and no_imp >= CFG["patience"]:
            print(f"early stop @ epoch {e}")
            break
    print(f"epoch {e}: trn_loss={float(tl):.4f} trn_mIoU={float(tm):.2f} val_mIoU={float(vm):.2f} best={best:.2f}")
(OUT / f"history_{shot}shot.json").write_text(json.dumps(hist))
print("saved", OUT / f"best_model_{shot}shot.pt")
del model, opt, tr, va
torch.cuda.empty_cache()


In [ ]:
# 10
shot = 5
print(f"=== pascal fold0 shot={shot} ===")
model = ProtoFormer(shot=shot, reduce_dim=CFG["reduce_dim"], checkpoint_path=SAM2_CKPT).to(device)
opt = optim.Adam(model.parameters(), lr=CFG["lr"])
tr = FSSDataset.build_dataloader(CFG["benchmark"], CFG["bsz"], CFG["nworker"], CFG["fold"], "trn", shot=shot)
va = FSSDataset.build_dataloader(CFG["benchmark"], 1, CFG["nworker"], CFG["fold"], "val", shot=shot)
hist = {"epoch": [], "trn_loss": [], "trn_miou": [], "val_miou": []}
best = -1.0
no_imp = 0
for e in range(CFG["niter"]):
    tl, tm, tf = train_epoch(e, model, tr, opt, True, amp, scaler)
    hist["epoch"].append(e); hist["trn_loss"].append(float(tl)); hist["trn_miou"].append(float(tm))
    hist["val_miou"].append(None)
    if (e + 1) % CFG.get("val_interval", 3) != 0 and e != CFG["niter"] - 1:
        print(f"epoch {e}: trn_loss={float(tl):.4f} trn_mIoU={float(tm):.2f}")
        continue
    with torch.no_grad():
        vl, vm, vf = train_epoch(e, model, va, opt, False, amp, None)
    hist["val_miou"][-1] = float(vm)
    if float(vm) > best:
        best = float(vm)
        no_imp = 0
        torch.save(model.state_dict(), OUT / f"best_model_{shot}shot.pt")
    else:
        no_imp += 1
        if CFG.get("patience", 20) > 0 and no_imp >= CFG["patience"]:
            print(f"early stop @ epoch {e}")
            break
    print(f"epoch {e}: trn_loss={float(tl):.4f} trn_mIoU={float(tm):.2f} val_mIoU={float(vm):.2f} best={best:.2f}")
(OUT / f"history_{shot}shot.json").write_text(json.dumps(hist))
print("saved", OUT / f"best_model_{shot}shot.pt")
del model, opt, tr, va
torch.cuda.empty_cache()


In [ ]:
# 11
def train_epoch(epoch, model, dl, opt, training, amp_dtype, scaler):
    utils.fix_randseed(None) if training else utils.fix_randseed(0)
    model.train() if training else model.eval()
    m = AverageMeter(dl.dataset)
    for i, b in enumerate(dl):
        b = utils.to_cuda(b)
        if training:
            with amp_context(amp_dtype):
                out = model(b["query_img"], b["support_imgs"], b["support_masks"])
                loss = model.compute_loss(out, b["query_mask"])
            opt.zero_grad()
            if scaler is not None:
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss.backward(); opt.step()
        else:
            with torch.no_grad(), amp_context(amp_dtype):
                out = model.predict_mask(b)
        pred = (out > 0.5).float().squeeze(1)
        inter, union = Evaluator.classify_prediction(pred, b)
        m.update(inter, union, b["class_id"], loss.detach().clone() if training else None)
        m.write_process(i, len(dl), epoch, write_batch_idx=100)
    m.write_result("Training" if training else "Validation", epoch)
    return utils.mean(m.loss_buf), *m.compute_iou()


In [ ]:
# 12
test_hist = {}
for shot in CFG["shots"]:
    print(f"=== pascal fold0 shot={shot} test ===")
    model = ProtoFormer(shot=shot, reduce_dim=CFG["reduce_dim"], checkpoint_path=SAM2_CKPT).to(device)
    model.load_state_dict(torch.load(OUT / f"best_model_{shot}shot.pt", map_location=device, weights_only=False))
    model.eval()
    va = FSSDataset.build_dataloader(CFG["benchmark"], 1, CFG["nworker"], CFG["fold"], "val", shot=shot)
    utils.fix_randseed(0)
    m = AverageMeter(va.dataset)
    for i, b in enumerate(va):
        b = utils.to_cuda(b)
        with torch.no_grad(), amp_context(amp):
            out = model.predict_mask(b)
        pred = (out > 0.5).float().squeeze(1)
        inter, union = Evaluator.classify_prediction(pred.clone(), b)
        m.update(inter, union, b["class_id"], loss=None)
        if i % 100 == 0:
            miou, fb = m.compute_iou()
            print(f"test {i}/{len(va)} mIoU={float(miou):.2f} FB={float(fb):.2f}")
    miou, fb = m.compute_iou()
    test_hist[f"pascal_fold0_{shot}shot"] = {"mIoU": float(miou), "FB_IoU": float(fb), "cfg": CFG}
    del model, va
    torch.cuda.empty_cache()
(OUT / "test_history.json").write_text(json.dumps(test_hist, indent=2))
print(test_hist)


In [ ]:
# 13
files = sorted(p.name for p in OUT.iterdir() if p.is_file())
print(files)
for n in files:
    print(n, (OUT / n).stat().st_size // 1024 // 1024, "MB")
with zipfile.ZipFile(OUT / "protoformer_out.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for n in files:
        if not n.endswith(".zip"):
            z.write(OUT / n, n)
print("saved:", OUT / "protoformer_out.zip")
